# Data Curation — CYP450 Direct Inhibition Train/Test

Builds the clean, trustworthy base dataset everything downstream (feature/descriptor
generation, CV fold construction, cluster-based splitting, modelling) will use.

**This notebook does:**
- loads only the two files settled by the schema audit (`00_schema_audit.ipynb`)
- checks for salts / multi-component SMILES, before touching the SMILES any other way
- canonicalizes SMILES and generates InChIKeys, via a shared function in `src/features.py`
- checks for duplicate compounds within train, within test, and leakage between them, by InChIKey
- re-confirms per-isoform label counts against the schema audit's verified figures
- saves a curated train/test file: identifiers, canonical SMILES, InChIKey, original labels unchanged

**This notebook does NOT do:** feature/descriptor generation for modelling, EDA beyond
what's needed to sanity-check curation, CV fold construction, cluster-based splitting,
or modelling. Its job is to report what's in the data and flag anything that needs a
decision — not to make modelling decisions itself.

Per `CLAUDE.md`: row counts are logged before/after every step as printed output, not
just in code; any undecided modelling choice is flagged and asked about rather than
silently picked; canonicalization/InChIKey logic lives in one shared module
(`src/features.py`), not duplicated inline.

In [1]:
import os

import pandas as pd
import numpy as np

from src.features import add_canonical_smiles_and_inchikey

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

RAW = 'data/raw'  # relative to repo root -- run this notebook with cwd = repo root
PROCESSED = 'data/processed'

## 1. Load and report shape/columns

Per the schema audit and task brief, only `cyp-challenge-TRAIN_inhibition.csv` and
`cyp-challenge-TEST-BLINDED.csv` are loaded. `TRAIN_TDI.csv` (confirmed byte-identical
pass-through for direct_inhibition), `single-concentration-TRAIN.csv`, and
`TRAIN_Emax.csv` are out of scope here and are not touched — that was settled by the
schema audit and is not reopened in this notebook.

In [2]:
inh = pd.read_csv(f'{RAW}/cyp-challenge-TRAIN_inhibition.csv')
print('TRAIN_inhibition.csv shape:', inh.shape)
print('columns:')
for c in inh.columns:
    print(' -', c)

TRAIN_inhibition.csv shape: (4905, 18)
columns:
 - Molecule_Name
 - SMILES
 - CYP1A2_pIC50_direct_inhibition
 - CYP2C9_pIC50_direct_inhibition
 - CYP2D6_pIC50_direct_inhibition
 - CYP3A4_pIC50_direct_inhibition
 - CYP1A2_pIC50_direct_inhibition_conf_high
 - CYP2C9_pIC50_direct_inhibition_conf_high
 - CYP2D6_pIC50_direct_inhibition_conf_high
 - CYP3A4_pIC50_direct_inhibition_conf_high
 - CYP1A2_pIC50_direct_inhibition_conf_low
 - CYP2C9_pIC50_direct_inhibition_conf_low
 - CYP2D6_pIC50_direct_inhibition_conf_low
 - CYP3A4_pIC50_direct_inhibition_conf_low
 - CYP1A2_pIC50_direct_inhibition_std
 - CYP2C9_pIC50_direct_inhibition_std
 - CYP2D6_pIC50_direct_inhibition_std
 - CYP3A4_pIC50_direct_inhibition_std


In [3]:
blinded = pd.read_csv(f'{RAW}/cyp-challenge-TEST-BLINDED.csv')
print('TEST-BLINDED.csv shape:', blinded.shape)
print('columns:')
for c in blinded.columns:
    print(' -', c)

TEST-BLINDED.csv shape: (750, 2)
columns:
 - Molecule_Name
 - SMILES


Both shapes match the schema audit exactly: TRAIN_inhibition.csv is 4,905 rows x
18 columns, TEST-BLINDED.csv is 750 rows x 2 columns (`Molecule_Name`, `SMILES` only —
no label columns, as expected for a blinded test set). Nothing unexpected at load
time.

## 2. Check for salts / multi-component structures

Done *before* any canonicalization or InChIKey generation, per the task brief — a `.`
in a SMILES string marks a disconnected fragment (a salt counter-ion or a mixture
component). This matters directly for step 5 (duplicate check by InChIKey): two rows
that are the same parent compound in different salt forms will NOT share an InChIKey
unless salts are stripped first. `cyp450.md`'s claim that real OpenADMET data doesn't
need salt-stripping is unverified for this specific file release, so it isn't assumed
here — only counted and reported.

Note: matching is done with `regex=False` (a literal `.`), not a regex `.` (which
would match "any character" and flag nearly every row).

In [4]:
def count_salts(df: pd.DataFrame, name: str) -> pd.DataFrame:
    is_multi = df['SMILES'].str.contains('.', regex=False, na=False)
    n = int(is_multi.sum())
    print(f'{name}: {n} of {len(df)} rows contain a "." in SMILES (disconnected fragment)')
    return df[is_multi]


train_salts = count_salts(inh, 'TRAIN_inhibition.csv')
test_salts = count_salts(blinded, 'TEST-BLINDED.csv')

TRAIN_inhibition.csv: 0 of 4905 rows contain a "." in SMILES (disconnected fragment)
TEST-BLINDED.csv: 0 of 750 rows contain a "." in SMILES (disconnected fragment)


In [5]:
train_salts[['Molecule_Name', 'SMILES']]

,Molecule_Name,SMILES


In [6]:
test_salts[['Molecule_Name', 'SMILES']]

,Molecule_Name,SMILES


**Finding: zero salts/multi-component SMILES in either file.** No row in
TRAIN_inhibition.csv or TEST-BLINDED.csv contains a `.` in its SMILES.

Per CLAUDE.md's "ask before a modelling decision that isn't explicitly specified" rule,
finding any salts here would require stopping to ask whether to strip them before
canonicalizing (step 3), since that's a decision this task brief explicitly declines to
make in advance. That question doesn't arise on this data release: there is nothing to
strip, so no salt-handling logic is needed in step 3 below, and the InChIKey duplicate
checks in steps 4-6 are not at risk of the salt-form false-negative described above.
This is a report of what's actually in this specific file, not a general claim about
the OpenADMET pipeline.

## 3. Canonicalize SMILES and generate InChIKeys

Uses the shared `canonicalize_smiles` / `smiles_to_inchikey` functions in
`src/features.py` (imported above), not inline logic — per CLAUDE.md's shared-module
rule and to avoid the cross-notebook drift already flagged as a risk in the schema
audit. Parse failures are surfaced, not silently dropped.

In [7]:
print('rows before canonicalization -- train:', len(inh), ' test:', len(blinded))

inh_c = add_canonical_smiles_and_inchikey(inh)
blinded_c = add_canonical_smiles_and_inchikey(blinded)

print('rows after canonicalization  -- train:', len(inh_c), ' test:', len(blinded_c))
print('(canonicalization adds columns only -- no rows dropped or added)')

rows before canonicalization -- train: 4905  test: 750
rows after canonicalization  -- train: 4905  test: 750
(canonicalization adds columns only -- no rows dropped or added)


In [8]:
train_fail = inh_c[inh_c['canonical_smiles'].isna()]
test_fail = blinded_c[blinded_c['canonical_smiles'].isna()]

print('train SMILES parse failures:', len(train_fail), 'of', len(inh_c))
print('test SMILES parse failures:', len(test_fail), 'of', len(blinded_c))

train SMILES parse failures: 0 of 4905
test SMILES parse failures: 0 of 750


In [9]:
train_fail[['Molecule_Name', 'SMILES']]

,Molecule_Name,SMILES


In [10]:
test_fail[['Molecule_Name', 'SMILES']]

,Molecule_Name,SMILES


**Finding: zero parse failures in either file.** Every SMILES in both
TRAIN_inhibition.csv and TEST-BLINDED.csv parses successfully with RDKit and has a
canonical SMILES and InChIKey. Nothing to surface for manual review here.

## 4. Duplicate check within the training set, by InChIKey

InChIKey, not canonical SMILES — matching OpenADMET's own PXR curation script and the
practice repo's prior convention. Flag and report only; no merging, dropping, or
resolving here — that's a separate, later decision.

In [11]:
train_ik_counts = inh_c.groupby('inchikey')['Molecule_Name'].nunique()
train_dup_ik = train_ik_counts[train_ik_counts > 1]

print('train: distinct InChIKeys:', inh_c['inchikey'].nunique(), 'of', len(inh_c), 'rows')
print('train: InChIKeys mapping to more than one Molecule_Name:', len(train_dup_ik))

if len(train_dup_ik):
    display(
        inh_c[inh_c['inchikey'].isin(train_dup_ik.index)]
        [['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey']]
        .sort_values('inchikey')
    )

train: distinct InChIKeys: 4905 of 4905 rows
train: InChIKeys mapping to more than one Molecule_Name: 0


**Finding: zero duplicate InChIKeys within the training set.** All 4,905 rows
have distinct InChIKeys — one InChIKey per Molecule_Name, no InChIKey shared across
more than one Molecule_Name. Nothing to flag for a merge/drop decision.

## 5. Duplicate check within the test set, by InChIKey

Same method as step 4, applied to TEST-BLINDED.csv.

In [12]:
test_ik_counts = blinded_c.groupby('inchikey')['Molecule_Name'].nunique()
test_dup_ik = test_ik_counts[test_ik_counts > 1]

print('test: distinct InChIKeys:', blinded_c['inchikey'].nunique(), 'of', len(blinded_c), 'rows')
print('test: InChIKeys mapping to more than one Molecule_Name:', len(test_dup_ik))

if len(test_dup_ik):
    display(
        blinded_c[blinded_c['inchikey'].isin(test_dup_ik.index)]
        [['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey']]
        .sort_values('inchikey')
    )

test: distinct InChIKeys: 750 of 750 rows
test: InChIKeys mapping to more than one Molecule_Name: 0


**Finding: zero duplicate InChIKeys within the test set.** All 750 rows have
distinct InChIKeys.

## 6. Train/test leakage check by InChIKey

Confirms whether any TEST-BLINDED.csv InChIKey also appears in TRAIN_inhibition.csv —
a stronger, more direct check than structural similarity, and worth surfacing
prominently if it turns up anything, since it would mean a test compound (in some SMILES
form) was already seen during training.

In [13]:
train_ik_set = set(inh_c['inchikey'].dropna())
test_ik_set = set(blinded_c['inchikey'].dropna())
leak_ik = train_ik_set & test_ik_set

print('train InChIKeys (non-null):', len(train_ik_set))
print('test InChIKeys (non-null):', len(test_ik_set))
print('InChIKeys present in BOTH train and test:', len(leak_ik))

if leak_ik:
    print()
    print('*** LEAKAGE DETECTED -- the following InChIKeys appear in both files: ***')
    for ik in sorted(leak_ik):
        print(' -', ik)

train InChIKeys (non-null): 4905
test InChIKeys (non-null): 750
InChIKeys present in BOTH train and test: 0


**Finding: zero leakage.** No InChIKey present in TEST-BLINDED.csv also appears
in TRAIN_inhibition.csv. No test compound (by InChIKey) has been seen in training.

## 7. Re-confirm per-isoform non-null counts against the schema audit

The schema audit's verified figures for TRAIN_inhibition.csv: CYP1A2 1,412 / CYP2C9
1,285 / CYP2D6 1,493 / CYP3A4 2,335 non-null pIC50 values. Checked here on the curated
(post-canonicalization) frame — canonicalization only adds columns, so these counts
should be unchanged from the raw file, but this is confirmed directly rather than
assumed. A mismatch would mean canonicalization somehow altered row alignment, which
would be a bug worth stopping for immediately.

In [14]:
expected_non_null = {
    'CYP1A2_pIC50_direct_inhibition': 1412,
    'CYP2C9_pIC50_direct_inhibition': 1285,
    'CYP2D6_pIC50_direct_inhibition': 1493,
    'CYP3A4_pIC50_direct_inhibition': 2335,
}

all_match = True
for col, expected_n in expected_non_null.items():
    actual_n = int(inh_c[col].notna().sum())
    match = actual_n == expected_n
    all_match = all_match and match
    print(f'{col}: expected {expected_n}, got {actual_n} -- {"MATCH" if match else "MISMATCH"}')

if not all_match:
    raise ValueError(
        "Per-isoform non-null counts do not match the schema audit's verified figures "
        "-- stopping per CLAUDE.md rather than proceeding on an unverified assumption."
    )

print()
print('All four isoform non-null counts match the schema audit exactly.')

CYP1A2_pIC50_direct_inhibition: expected 1412, got 1412 -- MATCH
CYP2C9_pIC50_direct_inhibition: expected 1285, got 1285 -- MATCH
CYP2D6_pIC50_direct_inhibition: expected 1493, got 1493 -- MATCH
CYP3A4_pIC50_direct_inhibition: expected 2335, got 2335 -- MATCH

All four isoform non-null counts match the schema audit exactly.


**Finding: all four isoform counts match the schema audit exactly.**
Canonicalization did not alter row alignment or drop/duplicate any rows.

## 8. Save curated output

Columns: `Molecule_Name`, original `SMILES`, `canonical_smiles`, `inchikey`, then all
original label columns unchanged (train only — the blinded test set has no label
columns to carry through). Row counts are logged before and after writing, and
confirmed again by reading the files back from disk.

In [15]:
os.makedirs(PROCESSED, exist_ok=True)

train_label_cols = [c for c in inh_c.columns if c not in ('Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey')]
train_curated = inh_c[['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey'] + train_label_cols]
test_curated = blinded_c[['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey']]

print('rows before save -- train:', len(inh_c), ' test:', len(blinded_c))
print('rows to write     -- train:', len(train_curated), ' test:', len(test_curated))
print()
print('train_curated columns:', list(train_curated.columns))
print('test_curated columns: ', list(test_curated.columns))

rows before save -- train: 4905  test: 750
rows to write     -- train: 4905  test: 750

train_curated columns: ['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey', 'CYP1A2_pIC50_direct_inhibition', 'CYP2C9_pIC50_direct_inhibition', 'CYP2D6_pIC50_direct_inhibition', 'CYP3A4_pIC50_direct_inhibition', 'CYP1A2_pIC50_direct_inhibition_conf_high', 'CYP2C9_pIC50_direct_inhibition_conf_high', 'CYP2D6_pIC50_direct_inhibition_conf_high', 'CYP3A4_pIC50_direct_inhibition_conf_high', 'CYP1A2_pIC50_direct_inhibition_conf_low', 'CYP2C9_pIC50_direct_inhibition_conf_low', 'CYP2D6_pIC50_direct_inhibition_conf_low', 'CYP3A4_pIC50_direct_inhibition_conf_low', 'CYP1A2_pIC50_direct_inhibition_std', 'CYP2C9_pIC50_direct_inhibition_std', 'CYP2D6_pIC50_direct_inhibition_std', 'CYP3A4_pIC50_direct_inhibition_std']
test_curated columns:  ['Molecule_Name', 'SMILES', 'canonical_smiles', 'inchikey']


In [16]:
train_path = f'{PROCESSED}/train_inhibition_curated.csv'
test_path = f'{PROCESSED}/test_blinded_curated.csv'

train_curated.to_csv(train_path, index=False)
test_curated.to_csv(test_path, index=False)

print('wrote', train_path)
print('wrote', test_path)

wrote data/processed/train_inhibition_curated.csv
wrote data/processed/test_blinded_curated.csv


In [17]:
reread_train = pd.read_csv(train_path)
reread_test = pd.read_csv(test_path)

print('rows after reading back from disk -- train:', len(reread_train), ' test:', len(reread_test))
assert len(reread_train) == len(train_curated), 'train row count changed on round-trip to disk'
assert len(reread_test) == len(test_curated), 'test row count changed on round-trip to disk'
print('row counts confirmed unchanged after round-trip to disk.')

rows after reading back from disk -- train: 4905  test: 750
row counts confirmed unchanged after round-trip to disk.


## 9. Final summary

**What was checked:**
1. Load + shape/column audit of both files — matched the schema audit exactly.
2. Salt / multi-component SMILES check — **0 found** in train, **0 found** in test.
3. Canonicalization + InChIKey generation via `src/features.py` — **0 parse failures**
   in either file.
4. Duplicate check within train by InChIKey — **0 InChIKeys** map to more than one
   `Molecule_Name`.
5. Duplicate check within test by InChIKey — **0 InChIKeys** map to more than one
   `Molecule_Name`.
6. Train/test leakage check by InChIKey — **0 InChIKeys** shared between train and
   test.
7. Per-isoform non-null counts on curated training data — **all four match** the
   schema audit's verified figures (CYP1A2 1,412 / CYP2C9 1,285 / CYP2D6 1,493 / CYP3A4
   2,335).
8. Curated output saved to `data/processed/train_inhibition_curated.csv` and
   `data/processed/test_blinded_curated.csv`, row counts confirmed unchanged on a
   disk round-trip.

**What was found:** a clean release. No salts, no parse failures, no within-file
duplicates, no train/test leakage. Nothing required a modelling decision on this run —
the salt-stripping question the task brief flagged as needing a stop-and-ask never
came up in practice, because the count was zero.

**What's still open / explicitly NOT done here:**
- No feature/descriptor generation, CV fold construction, cluster-based splitting, or
  modelling — out of scope for this notebook, deferred to later notebooks.
- No EDA beyond the structural sanity checks above.
- Because every check in steps 2, 4, 5, and 6 came back empty, none of the "flag but
  don't resolve" items from the task brief actually materialized into open decisions
  on this data release — there is nothing pending to decide before the next notebook.
  If a future data refresh changes any of these findings, this notebook's checks
  should be re-run rather than assumed still clean.

## Before finishing, check against CLAUDE.md directly

- **Row counts logged before/after every step:** yes — load (section 1), canonicalization
  (section 3), and save (section 8) all print row counts, including a disk round-trip
  confirmation.
- **Any undecided choice flagged and asked about, not silently picked:** the one
  modelling decision this task brief called out in advance (salt-stripping) was
  checked, reported, and found not to apply (0 salts) — nothing was silently decided.
- **Shared canonicalization/InChIKey logic lives in `src/features.py`, not duplicated
  inline:** yes — `canonicalize_smiles`, `smiles_to_inchikey`, and
  `add_canonical_smiles_and_inchikey` are defined there and imported in cell 2, not
  redefined in this notebook.
- **Every seed used is logged:** no randomness is introduced anywhere in this notebook
  (no sampling, no train/test splitting, no model fitting) — this rule doesn't apply
  here, confirmed rather than assumed.
- **Nothing here touches test-set labels:** confirmed in section 1 — `TEST-BLINDED.csv`
  has only `Molecule_Name` and `SMILES`, no label columns exist to touch.